# Week 8 Integrative Assignment: GreenLeaf's Loyalty Revolution

**ISM4641 - Python for Business Analytics**  
**Due Date:** March 8 2026  
**Points:** 100 points

---

## The Story

**Thursday, 9:00 AM**

Marcus Thompson settles into his desk at **GreenLeaf Markets**, a regional grocery chain with 12 stores across Central Florida. As the Marketing Analyst, he's been working on improving customer retention—a critical challenge in the competitive grocery industry.

His manager, Rachel Kim (VP of Marketing), walks over with a determined look.

*"Marcus, the board meeting is Monday. They want to see data on our loyalty program before they approve the budget for next year's expansion. We've been collecting customer data for six months, but nobody's done a deep analysis yet."*

*"I need you to answer some key questions: Who are our best customers? Which stores are performing well? Is our loyalty program actually working? The board wants numbers, not guesses."*

Marcus pulls up the data files. Transaction records, customer profiles, store information—thousands of rows across multiple tables. This is exactly the kind of challenge he's been preparing for.

*"I'll have the analysis ready by end of day tomorrow,"* he says.

---

## Your Mission

You are Marcus. Use Pandas to analyze GreenLeaf's customer and transaction data to prepare insights for the board presentation.

**Skills you'll apply:**
- Creating and manipulating DataFrames
- Merging/joining multiple datasets
- Filtering and selecting data
- Groupby operations and aggregation
- Adding calculated columns
- Sorting and ranking

---

## Assignment Requirements

1. Complete all sections of this notebook
2. Use Pandas operations for data manipulation
3. Include your video walkthrough (3-5 minutes)

**Video Walkthrough:** Present your findings as if briefing the VP of Marketing before the board meeting.

---

## Student Information

**Name:** Luis Vieira  
**USF ID:** U1873984  
**Date:** March 8 2026

In [ ]:
# Import libraries
import pandas as pd
import numpy as np

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set random seed for reproducibility
np.random.seed(8641)

# Generate Stores Data
stores = pd.DataFrame({
    'store_id': range(1, 13),
    'store_name': ['Downtown Tampa', 'Brandon', 'Clearwater', 'St. Petersburg',
                   'Wesley Chapel', 'Lakeland', 'Sarasota', 'Bradenton',
                   'Temple Terrace', 'Plant City', 'Riverview', 'Palm Harbor'],
    'region': ['Tampa Bay', 'Tampa Bay', 'Tampa Bay', 'Tampa Bay',
               'North', 'North', 'South', 'South',
               'Tampa Bay', 'North', 'Tampa Bay', 'Tampa Bay'],
    'size_sqft': [45000, 38000, 42000, 40000, 52000, 35000, 48000, 36000,
                  32000, 28000, 44000, 39000],
    'opened_year': [2015, 2017, 2016, 2018, 2022, 2019, 2020, 2021,
                    2018, 2020, 2021, 2019]
})

# Generate Customers Data
n_customers = 500
customers = pd.DataFrame({
    'customer_id': range(1001, 1001 + n_customers),
    'name': [f"Customer_{i}" for i in range(1, n_customers + 1)],
    'join_date': pd.date_range('2023-01-01', periods=n_customers, freq='12H').strftime('%Y-%m-%d'),
    'loyalty_tier': np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'],
                                      n_customers, p=[0.50, 0.30, 0.15, 0.05]),
    'home_store': np.random.choice(range(1, 13), n_customers),
    'age_group': np.random.choice(['18-25', '26-35', '36-45', '46-55', '56-65', '65+'],
                                   n_customers, p=[0.10, 0.25, 0.25, 0.20, 0.12, 0.08])
})

# Generate Transactions Data
n_transactions = 3000

# Higher tier customers shop more frequently
tier_weights = {'Bronze': 1, 'Silver': 1.5, 'Gold': 2, 'Platinum': 3}
customer_weights = customers['loyalty_tier'].map(tier_weights).values
customer_weights = customer_weights / customer_weights.sum()

transactions = pd.DataFrame({
    'transaction_id': range(1, n_transactions + 1),
    'customer_id': np.random.choice(customers['customer_id'], n_transactions, p=customer_weights),
    'store_id': np.random.choice(range(1, 13), n_transactions),
    'transaction_date': pd.date_range('2024-07-01', '2024-12-31', periods=n_transactions).strftime('%Y-%m-%d'),
    'items_purchased': np.random.randint(3, 35, n_transactions),
    'subtotal': np.round(np.random.uniform(15, 250, n_transactions), 2)
})

# Higher tier customers tend to spend more - adjust subtotals
tier_multiplier = customers.set_index('customer_id')['loyalty_tier'].map(
    {'Bronze': 1.0, 'Silver': 1.15, 'Gold': 1.3, 'Platinum': 1.5}
)
transactions['subtotal'] = transactions.apply(
    lambda x: round(x['subtotal'] * tier_multiplier.get(x['customer_id'], 1.0), 2), axis=1
)

# Add discount based on loyalty tier
transactions = transactions.merge(
    customers[['customer_id', 'loyalty_tier']], on='customer_id', how='left'
)
discount_rates = {'Bronze': 0.02, 'Silver': 0.05, 'Gold': 0.10, 'Platinum': 0.15}
transactions['discount'] = transactions.apply(
    lambda x: round(x['subtotal'] * discount_rates[x['loyalty_tier']], 2), axis=1
)
transactions['total'] = transactions['subtotal'] - transactions['discount']
transactions = transactions.drop('loyalty_tier', axis=1)

print("Data loaded successfully!")
print(f"\nStores: {len(stores)} locations")
print(f"Customers: {len(customers)} loyalty members")
print(f"Transactions: {len(transactions)} records (Jul-Dec 2024)")

print("\n" + "="*60)
print("STORES PREVIEW:")
display(stores.head())

print("\nCUSTOMERS PREVIEW:")
display(customers.head())

print("\nTRANSACTIONS PREVIEW:")
display(transactions.head())

---

## Part 1: Customer Segmentation Analysis (20 points)

Rachel's first question: *"Who are our customers? I need a breakdown by loyalty tier and demographics."*

**Your Task:**
1. Calculate the count and percentage of customers in each loyalty tier
2. Create a cross-tabulation of loyalty tier by age group
3. Find the average number of transactions per customer by loyalty tier
4. Calculate the average spend per transaction by loyalty tier
5. Identify the most common age group in each loyalty tier

In [11]:
# Part 1: Customer Segmentation Analysis

# Define logical tier order for sorting
tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']

# 1. Count and percentage by loyalty tier
tier_counts = customers['loyalty_tier'].value_counts().reindex(tier_order)
tier_pcts = (customers['loyalty_tier'].value_counts(normalize=True) * 100).round(2).reindex(tier_order)
tier_dist = pd.DataFrame({'Count': tier_counts, 'Percentage (%)': tier_pcts})

# 2. Cross-tabulation of tier by age group
tier_age_xtab = pd.crosstab(customers['loyalty_tier'], customers['age_group']).reindex(tier_order)

# 3. Average transactions per customer by tier
txn_counts = transactions['customer_id'].value_counts().reset_index()
txn_counts.columns = ['customer_id', 'txn_count']
customer_txns = customers.merge(txn_counts, on='customer_id', how='left').fillna(0)
avg_txns_by_tier = customer_txns.groupby('loyalty_tier')['txn_count'].mean().round(2).reindex(tier_order)

# 4. Average spend per transaction by tier
txn_merged = transactions.merge(customers[['customer_id', 'loyalty_tier']], on='customer_id', how='left')
avg_spend_by_tier = txn_merged.groupby('loyalty_tier')['total'].mean().round(2).reindex(tier_order)

# 5. Most common age group in each tier
most_common_age = tier_age_xtab.idxmax(axis=1)

# Display results
print("CUSTOMER SEGMENTATION ANALYSIS")
print("="*70)

print("\nLoyalty Tier Distribution:")
print("-"*70)
print(tier_dist)

print("\nTier by Age Group Cross-tabulation:")
print("-"*70)
print(tier_age_xtab)

print("\nCustomer Value by Tier:")
print("-"*70)
value_summary = pd.DataFrame({
    'Avg Transactions': avg_txns_by_tier,
    'Avg Spend per Txn': avg_spend_by_tier.apply(lambda x: f'${x:,.2f}'),
    'Top Age Group': most_common_age
})
print(value_summary)

CUSTOMER SEGMENTATION ANALYSIS

Loyalty Tier Distribution:
----------------------------------------------------------------------
              Count  Percentage (%)
loyalty_tier                       
Bronze          257            51.4
Silver          158            31.6
Gold             57            11.4
Platinum         28             5.6

Tier by Age Group Cross-tabulation:
----------------------------------------------------------------------
age_group     18-25  26-35  36-45  46-55  56-65  65+
loyalty_tier                                        
Bronze           34     66     57     52     26   22
Silver           22     36     40     38     13    9
Gold              7     17     13     11      4    5
Platinum          3      7      6      7      4    1

Customer Value by Tier:
----------------------------------------------------------------------
              Avg Transactions Avg Spend per Txn Top Age Group
loyalty_tier                                                  
Bronze




### Insights or Next Steps

*   **Targeted Retention**: Since Platinum and Gold members spend more and visit more frequently but represent only 17% of the base, Marcus should recommend loyalty programs specifically designed to move "Silver" members (the second-largest group) into "Gold" status.
*   **Age-Based Marketing**: Marketing campaigns for the Platinum tier should be tailored toward the 26-35 demographic, as they are the most frequent high-value customers.


---

## Part 2: Store Performance Analysis (20 points)

Rachel continues: *"The board will ask about store performance. Which locations are our stars, and which need attention?"*

**Your Task:**
1. Calculate total revenue, transaction count, and average basket size for each store
2. Merge store details with performance metrics
3. Calculate revenue per square foot for each store
4. Rank stores by total revenue
5. Compare performance by region (Tampa Bay vs North vs South)
6. Identify the top 3 and bottom 3 performing stores by revenue per square foot

In [15]:
# 1. Calculate store metrics from transactions
store_metrics = transactions.groupby('store_id').agg(
    total_revenue=('total', 'sum'),
    transaction_count=('transaction_id', 'count'),
    avg_basket_size=('total', 'mean')
).reset_index()

# 2. Merge with store details
store_performance = store_metrics.merge(stores, on='store_id', how='left')

# 3. Calculate revenue per square foot
store_performance['revenue_per_sqft'] = store_performance['total_revenue'] / store_performance['size_sqft']

# 4. Rank stores by total revenue
store_performance['revenue_rank'] = store_performance['total_revenue'].rank(ascending=False)

# 5. Regional performance comparison
regional_perf = store_performance.groupby('region').agg(
    total_revenue=('total_revenue', 'sum'),
    avg_rev_per_sqft=('revenue_per_sqft', 'mean')
).sort_values(by='total_revenue', ascending=False)

# 6. Top and bottom performers by revenue per sqft
top_3_stores = store_performance.nlargest(3, 'revenue_per_sqft')[['store_name', 'revenue_per_sqft']]
bottom_3_stores = store_performance.nsmallest(3, 'revenue_per_sqft')[['store_name', 'revenue_per_sqft']]

# Display results
print("STORE PERFORMANCE ANALYSIS")
print("="*80)

print("\nStore Performance Summary:")
print("-"*80)
# Reset index for better readability in the summary table
display(store_performance.sort_values(by='revenue_rank').reset_index(drop=True))

print("\nRegional Performance Comparison:")
print("-"*80)
print(regional_perf)

print("\nTop 3 Stores by Revenue per Sq Ft:")
print(top_3_stores)

print("\nBottom 3 Stores (Need Attention):")
print(bottom_3_stores)

STORE PERFORMANCE ANALYSIS

Store Performance Summary:
--------------------------------------------------------------------------------


,store_id,total_revenue,transaction_count,avg_basket_size,store_name,region,size_sqft,opened_year,revenue_per_sqft,revenue_rank
0,5,43791.96,286,153.118741,Wesley Chapel,North,52000,2022,0.842153,1.0
1,8,41692.00,285,146.287719,Bradenton,South,36000,2021,1.158111,2.0
2,2,37680.81,248,151.938750,Brandon,Tampa Bay,38000,2017,0.991600,3.0
3,3,37367.34,249,150.069639,Clearwater,Tampa Bay,42000,2016,0.889699,4.0
4,10,37139.94,263,141.216502,Plant City,North,28000,2020,1.326426,5.0
5,9,36057.75,260,138.683654,Temple Terrace,Tampa Bay,32000,2018,1.126805,6.0
6,1,35948.16,243,147.934815,Downtown Tampa,Tampa Bay,45000,2015,0.798848,7.0
7,12,35227.25,262,134.455153,Palm Harbor,Tampa Bay,39000,2019,0.903263,8.0
8,7,33827.90,231,146.441126,Sarasota,South,48000,2020,0.704748,9.0
9,11,33234.37,225,147.708311,Riverview,Tampa Bay,44000,2021,0.755327,10.0



Regional Performance Comparison:
--------------------------------------------------------------------------------
           total_revenue  avg_rev_per_sqft
region                                    
Tampa Bay      244489.16          0.884268
North          114082.30          1.038578
South           75519.90          0.931430

Top 3 Stores by Revenue per Sq Ft:
       store_name  revenue_per_sqft
9      Plant City          1.326426
7       Bradenton          1.158111
8  Temple Terrace          1.126805

Bottom 3 Stores (Need Attention):
        store_name  revenue_per_sqft
6         Sarasota          0.704748
3   St. Petersburg          0.724337
10       Riverview          0.755327



### Insights or Next Steps

*   **Space Optimization Strategy:** Marcus should investigate the operational practices at Plant City to see if they can be replicated at larger, less efficient stores like Sarasota or St. Petersburg to improve their revenue per square foot.
*   **Regional Resource Allocation:** Since the North region shows the highest efficiency, the board should consider expansion opportunities or increased marketing spend in that region to capitalize on its high ROI relative to store size.


---

## Part 3: Loyalty Program ROI (25 points)

*"Here's the big question,"* Rachel says. *"Is our loyalty program worth the investment? We're giving discounts to higher tiers—are we getting enough in return?"*

**Your Task:**
1. Calculate total discounts given by loyalty tier
2. Calculate total revenue (after discounts) generated by each tier
3. Calculate the "discount ratio" (discounts / subtotal) for each tier
4. Calculate Customer Lifetime Value (CLV) proxy: average total spending per customer by tier
5. Analyze whether higher-tier customers visit more frequently (transactions per customer)
6. Calculate the "loyalty lift": how much more do Gold/Platinum spend vs Bronze per transaction?

In [17]:
# Part 3: Loyalty Program ROI

# Merge transactions with customer tier information
roi_merged = transactions.merge(customers[['customer_id', 'loyalty_tier']], on='customer_id', how='left')
tier_order = ['Bronze', 'Silver', 'Gold', 'Platinum']

# 1 & 2. Total discounts and revenue (after discounts) by tier
tier_financials = roi_merged.groupby('loyalty_tier').agg(
    total_subtotal=('subtotal', 'sum'),
    total_discounts=('discount', 'sum'),
    total_revenue=('total', 'sum'),
    txn_count=('transaction_id', 'count')
).reindex(tier_order)

# 3. Calculate the discount ratio by tier
tier_financials['discount_ratio'] = (tier_financials['total_discounts'] / tier_financials['total_subtotal']).round(4)

# 4. CLV proxy - average total spending per customer by tier
cust_spend = roi_merged.groupby(['customer_id', 'loyalty_tier'])['total'].sum().reset_index()
avg_clv_proxy = cust_spend.groupby('loyalty_tier')['total'].mean().reindex(tier_order)

# 5. Transaction frequency (transactions per customer) by tier
cust_counts = roi_merged.groupby(['customer_id', 'loyalty_tier'])['transaction_id'].count().reset_index()
avg_freq = cust_counts.groupby('loyalty_tier')['transaction_id'].mean().reindex(tier_order)

# 6. Loyalty lift: Compare Gold/Platinum spend vs Bronze per transaction
avg_spend_per_txn = tier_financials['total_revenue'] / tier_financials['txn_count']
bronze_avg = avg_spend_per_txn['Bronze']
loyalty_lift = (avg_spend_per_txn / bronze_avg - 1) * 100

# Display results
print("LOYALTY PROGRAM ROI ANALYSIS")
print("="*70)

print("\nFinancial Summary by Tier:")
print("-"*70)
financial_display = tier_financials[['total_discounts', 'total_revenue', 'discount_ratio']].copy()
financial_display['total_discounts'] = financial_display['total_discounts'].apply(lambda x: f'${x:,.2f}')
financial_display['total_revenue'] = financial_display['total_revenue'].apply(lambda x: f'${x:,.2f}')
print(financial_display)

print("\nCustomer Behavior by Tier:")
print("-"*70)
behavior_summary = pd.DataFrame({
    'Avg CLV Proxy': avg_clv_proxy.apply(lambda x: f'${x:,.2f}'),
    'Avg Annual Visits': avg_freq.round(2)
})
print(behavior_summary)

print("\nLoyalty Lift Analysis (vs Bronze %):")
print("-"*70)
print(loyalty_lift.round(2).apply(lambda x: f'{x}%'))

# Summary Insights for Part 3
print("\nSUMMARY OF ROI FINDINGS:")
print("-"*70)
print(f"1. High-Value Conversion: Platinum customers visit {avg_freq['Platinum']/avg_freq['Bronze']:.1f}x more often than Bronze members.")
print(f"2. Spend Premium: Gold and Platinum members spend {loyalty_lift['Gold']:.1f}% and {loyalty_lift['Platinum']:.1f}% more per visit respectively.")
print(f"3. Efficiency: Despite higher discount ratios ({tier_financials.loc['Platinum', 'discount_ratio']*100:.0f}% for Platinum), the volume and basket size more than offset the cost.")

LOYALTY PROGRAM ROI ANALYSIS

Financial Summary by Tier:
----------------------------------------------------------------------
             total_discounts total_revenue  discount_ratio
loyalty_tier                                              
Bronze             $3,108.89   $152,342.52            0.02
Silver             $7,697.87   $146,255.43            0.05
Gold               $8,210.34    $73,891.10            0.10
Platinum          $10,870.95    $61,602.31            0.15

Customer Behavior by Tier:
----------------------------------------------------------------------
             Avg CLV Proxy  Avg Annual Visits
loyalty_tier                                 
Bronze             $597.42               4.52
Silver             $925.67               6.30
Gold             $1,296.34               8.53
Platinum         $2,200.08              13.11

Loyalty Lift Analysis (vs Bronze %):
----------------------------------------------------------------------
loyalty_tier
Bronze        0.0%
Si

### Insights or Next Steps

*   **High-Tier Incentivization:** The data confirms a significant 'Loyalty Lift,' with Platinum members spending nearly 27% more per transaction than Bronze members. Marcus should recommend protecting these high-value margins by ensuring the 15% discount continues to be offset by high-volume traffic.
*   **Conversion Strategy:** Since Gold members show similar spending patterns to Platinum but with lower visit frequency, a targeted 'Road to Platinum' campaign could be implemented to increase visit frequency among the Gold demographic to match Platinum behavior.

---

## Part 4: Customer Retention Insights (20 points)

Rachel has one more concern: *"Are customers staying engaged? I want to know who our VIPs are and if we're at risk of losing anyone."*

**Your Task:**
1. Identify "VIP" customers: top 10% by total spending
2. Create a customer summary with: total spent, transaction count, average basket, first purchase date, last purchase date
3. Calculate "recency" for each customer (days since last transaction, assuming today is 2025-01-15)
4. Identify "at-risk" customers: those who haven't purchased in 60+ days
5. Cross-reference at-risk customers with loyalty tier—are we losing valuable customers?
6. Find the stores where VIP customers shop most frequently

In [18]:
import pandas as pd

# 1. Ensure transaction_date is datetime and define reference date
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])
today = pd.to_datetime('2025-01-15')

# 2. Create customer summary by grouping transactions
customer_summary = transactions.groupby('customer_id').agg(
    total_spent=('total', 'sum'),
    transaction_count=('transaction_id', 'count'),
    first_purchase=('transaction_date', 'min'),
    last_purchase=('transaction_date', 'max')
).reset_index()

# 3. Merge with customers and calculate avg_basket
customer_summary = customer_summary.merge(customers[['customer_id', 'loyalty_tier', 'name']], on='customer_id', how='left')
customer_summary['avg_basket'] = (customer_summary['total_spent'] / customer_summary['transaction_count']).round(2)

# 4. Calculate recency (days since last purchase)
customer_summary['recency'] = (today - customer_summary['last_purchase']).dt.days

# 5. Identify VIP customers (top 10% by total spending)
spend_threshold = customer_summary['total_spent'].quantile(0.90)
customer_summary['is_vip'] = customer_summary['total_spent'] >= spend_threshold
vip_customers = customer_summary[customer_summary['is_vip']].sort_values(by='total_spent', ascending=False)

# 6. Identify at-risk customers (60+ days since last purchase)
at_risk_customers = customer_summary[customer_summary['recency'] >= 60].copy()

# 7. At-risk customers by loyalty tier
at_risk_by_tier = at_risk_customers.groupby('loyalty_tier').size().reindex(['Bronze', 'Silver', 'Gold', 'Platinum']).fillna(0).astype(int)

# 8. Identify stores where VIPs shop most frequently
vip_ids = vip_customers['customer_id']
vip_txns = transactions[transactions['customer_id'].isin(vip_ids)].merge(stores[['store_id', 'store_name']], on='store_id')
vip_store_pref = vip_txns['store_name'].value_counts().head(5)

# 9. Display results
print("CUSTOMER RETENTION INSIGHTS")
print("="*70)

print(f"\nVIP Customer Profile (Top 10% - Threshold: ${spend_threshold:,.2f}):")
print("-"*70)
display(vip_customers[['name', 'loyalty_tier', 'total_spent', 'transaction_count', 'recency']].head(10))

print("\nAt-Risk Customer Analysis (60+ Days Since Last Purchase):")
print("-"*70)
print(f"Total At-Risk Customers: {len(at_risk_customers)}")
print("At-Risk Breakdown by Tier:")
print(at_risk_by_tier)

print("\nVIP Preferred Stores (Top 5 by Transaction Volume):")
print("-"*70)
print(vip_store_pref)


CUSTOMER RETENTION INSIGHTS

VIP Customer Profile (Top 10% - Threshold: $1,622.88):
----------------------------------------------------------------------


,name,loyalty_tier,total_spent,transaction_count,recency
291,Customer_292,Platinum,4089.25,25,19
486,Customer_489,Platinum,4074.02,23,21
180,Customer_181,Platinum,3354.61,22,25
303,Customer_304,Platinum,2921.54,19,21
360,Customer_361,Platinum,2751.34,17,31
167,Customer_168,Platinum,2656.28,15,34
461,Customer_463,Platinum,2649.74,12,22
258,Customer_259,Platinum,2587.84,14,19
235,Customer_236,Platinum,2500.02,12,40
492,Customer_495,Platinum,2416.66,14,34



At-Risk Customer Analysis (60+ Days Since Last Purchase):
----------------------------------------------------------------------
Total At-Risk Customers: 126
At-Risk Breakdown by Tier:
loyalty_tier
Bronze      89
Silver      29
Gold         7
Platinum     1
dtype: int64

VIP Preferred Stores (Top 5 by Transaction Volume):
----------------------------------------------------------------------
store_name
Temple Terrace    69
Clearwater        65
Bradenton         62
Wesley Chapel     61
Palm Harbor       57
Name: count, dtype: int64


### Strategic Retention Insights

**VIP Customer Profile & Spending Threshold**
 top 10% of GreenLeaf customers have a spending threshold of approximately **$1,622.88**. This segment is heavily dominated by Platinum and Gold tier members, who represent the highest lifetime value for our loyalty program. Maintaining high engagement within this group is essential for sustaining our current revenue growth.

**High-Priority Re-engagement Targets**
While the majority of 'At-Risk' customers (those who haven't shopped in 60+ days) are in the Bronze tier, there are **1 Platinum and 7 Gold members** currently in this category. These 8 individuals represent a significant loss of potential revenue and should be Marcus's highest priority for personalized outreach, such as exclusive 'We Miss You' offers or bonus loyalty points.

**Store-Specific Engagement Strategies**
To maximize the impact of VIP-focused initiatives, Marcus should prioritize **Temple Terrace and Clearwater** for loyalty events or VIP-only offers. These stores are the most frequented locations for our top-tier members, making them the ideal testing grounds for premium customer experience enhancements.

**Churn Prevention & Recency Monitoring**
Moving forward, the 'recency' metric should be integrated into our monthly dashboard. By serving as an early warning system, monitoring days since the last purchase will allow the marketing team to intervene before high-value customers cross the 60-day at-risk threshold, effectively reducing long-term churn.

---

## Part 5: Board Presentation Summary (15 points)

It's Friday afternoon. Marcus needs to compile his findings into a clear executive summary.

**Your Task:**
Create a comprehensive board summary that includes:
1. Customer base overview (total members, tier distribution)
2. Revenue summary (total revenue, average transaction value)
3. Top performing store and region
4. Loyalty program ROI verdict (is it working?)
5. Customer retention status (VIPs count, at-risk count)
6. Three actionable recommendations based on your analysis

In [34]:
print("\n" + "="*70)
print("                 GREENLEAF MARKETS")
print("         LOYALTY PROGRAM ANALYSIS - BOARD SUMMARY")
print("              Prepared by: Luis Vieira")
print("              Analysis Period: Jul-Dec 2024")
print("="*70)

# 1. Customer Base Overview
print("\n1. CUSTOMER BASE OVERVIEW")
print("-"*70)
print(f"Total Loyalty Members: {len(customers)}")
print("\nTier Distribution:")
print(tier_dist)

# 2. Revenue Summary
total_rev = transactions['total'].sum()
total_txns = len(transactions)
avg_basket = transactions['total'].mean()

print("\n2. REVENUE SUMMARY")
print("-"*70)
print(f"Total Revenue: ${total_rev:,.2f}")
print(f"Total Transactions: {total_txns:,}")
print(f"Average Basket Size: ${avg_basket:,.2f}")

# 3. Store Performance Highlight
top_region = regional_perf.index[0]
top_efficiency_store = top_3_stores.iloc[0]

print("\n3. STORE PERFORMANCE HIGHLIGHTS")
print("-"*70)
print(f"Top Performing Region: {top_region}")
print(f"Efficiency Leader: {top_efficiency_store['store_name']} (${top_efficiency_store['revenue_per_sqft']:,.2f} per sqft)")

# 4. Loyalty Program ROI Verdict
print("\n4. LOYALTY PROGRAM ROI ASSESSMENT")
print("-"*70)
print("Verdict: SUCCESSFUL INVESTMENT")
print(f"- Platinum members generate {avg_freq['Platinum']/avg_freq['Bronze']:.1f}x more visits than Bronze.")
print(f"- Loyalty Lift: Platinum spend per txn is {loyalty_lift['Platinum']:.1f}% higher than base.")

# 5. Customer Retention Status
vip_count = len(vip_customers)
at_risk_count = at_risk_by_tier.sum()

print("\n5. CUSTOMER RETENTION STATUS")
print("-"*70)
print(f"VIP Customers (Top 10%): {vip_count}")
print(f"At-Risk Customers (60+ days): {at_risk_count}")
print(f"High-Value Risk: {at_risk_by_tier['Gold'] + at_risk_by_tier['Platinum']} Gold/Platinum members need urgent outreach.")

# 6. Recommendations
print("\n6. STRATEGIC RECOMMENDATIONS")
print("-"*70)
print("1. Implement 'Road to Platinum' campaign for Gold members to increase visit frequency.")
print("2. Deploy automated re-engagement triggers for Silver/Gold members reaching 45 days of inactivity.")
print("3. Audit Plant City operations to create an efficiency blueprint for low-performing South region stores.")

print("\n" + "="*70)
print("                    END OF REPORT")
print("="*70)


                 GREENLEAF MARKETS
         LOYALTY PROGRAM ANALYSIS - BOARD SUMMARY
              Prepared by: Luis Vieira
              Analysis Period: Jul-Dec 2024

1. CUSTOMER BASE OVERVIEW
----------------------------------------------------------------------
Total Loyalty Members: 500

Tier Distribution:
              Count  Percentage (%)
loyalty_tier                       
Bronze          257            51.4
Silver          158            31.6
Gold             57            11.4
Platinum         28             5.6

2. REVENUE SUMMARY
----------------------------------------------------------------------
Total Revenue: $434,091.36
Total Transactions: 3,000
Average Basket Size: $144.70

3. STORE PERFORMANCE HIGHLIGHTS
----------------------------------------------------------------------
Top Performing Region: Tampa Bay
Efficiency Leader: Plant City ($1.33 per sqft)

4. LOYALTY PROGRAM ROI ASSESSMENT
----------------------------------------------------------------------
Verdict


### Data Analysis Key Findings

*   **Loyalty Tier Distribution**: The program is comprised of 500 members, with the majority in Bronze (51.4%) and Silver (31.6%), while high-value Gold and Platinum tiers make up 11.4% and 5.6% respectively.
*   **Revenue Metrics**: Total revenue for the analysis period (Jul-Dec 2024) reached \$434,091.36 from 3,000 transactions, maintaining an average basket size of \$144.70.
*   **Operational Excellence**: The Tampa Bay region is the revenue leader, while the Plant City store serves as the efficiency benchmark with a revenue of \$1.33 per square foot.
*   **Loyalty ROI**: Platinum members demonstrate significantly higher engagement, visiting 2.9x more frequently than Bronze members and spending 26.9% more per transaction.
*   **Customer Retention Risk**: While there are 50 VIP customers (top 10% spenders), 126 members are "at-risk" (60+ days since last purchase), including 8 critical Gold/Platinum members.



---

## Epilogue

**Monday, 4:30 PM**

The board meeting ended an hour ago. Rachel stops by Marcus's desk, a rare smile on her face.

*"The board loved it. They approved the loyalty program budget for next year—actually increased it by 15%. Your analysis showing the ROI from Platinum customers was exactly what they needed to see."*

*"They also approved my proposal for a new position: Senior Marketing Analyst focused on customer analytics. I'm recommending you for it. Interested?"*

Marcus grins. The hours spent learning Pandas just paid off in ways he never expected. *"Absolutely. When do I start?"*

---

## Interaction Log

### AI Tool Usage
*Did you use any AI tools? If so, what for?*

[Yes, I utilized Gemini colab copilot agent for code implementation, edits, and main insights]

### Pandas Operations Used
*List the key Pandas operations you used and briefly explain why.*

[Merge, GroupBy,Crosstab,]

### Verification Steps
*How did you verify your calculations? Did you spot-check any results?*

[Evaluated validity of tables and numeric results by looking at exectuions panel after each generated output]

---

## Submission Checklist

- [X] All code cells run without errors
- [X] All five parts are completed
- [X] Pandas operations used for data manipulation
- [X] Merges and groupby operations correctly applied
- [X] Board summary is complete and professional
- [X] Interaction log is completed
- [X] Video walkthrough is recorded (3-5 minutes)

---
*ISM4641 Python for Business Analytics - Week 8 Integrative Assignment*